# run03_v2dynamic2class_rot15（2026-10-27）
- 檔名規則：YOLOv8n_runXX_資料版本_說明_日期.ipynb；每個版本使用獨立檔案，變更記錄寫在本區，結論寫在檔尾。
- 前一版：run02_v2dynamic2class_baseline_20261027（2 類 raw baseline，test mAP50 0.688；person 0.866、dynamic_obstacle 0.51；seed=0、無擴增）；本檔由 run02 另存副本，run02 不再更動。
- 資料：與 run02 完全相同（Roboflow alert-yolo-v1 version 2，標籤合併為 person / dynamic_obstacle；Train 499 / Valid 90 / Test 75）。valid / test 維持原始影像。
- 變更：只改一個因素——訓練時加入隨機旋轉 degrees=15（±15°，Ultralytics 內建即時擴增，只作用於訓練集，不產生離線副本）。其餘擴增（mosaic、hsv、fliplr、scale、translate）維持 0。
- 設定：其餘與 run02 完全相同（yolov8n.pt, imgsz=320, epochs=100, patience=20, batch=32, seed=0, deterministic, weight_decay=0.0005）。
- 用途：擴增實驗第一組（旋轉）；與 run02 在同一份 test 比較，以 person 與 dynamic_obstacle 為主。後續依序 ±30°、±45°，再做 HSV。
- 注意：run02 僅 seed=0 一次、尚無變異估計，差距在約 0.03 以內不宜解讀為有效；dynamic_obstacle 的 test 僅 31 個實例，結論暫定。
- 輸出位置：Google Drive/capstone_yolo/run03_v2dynamic2class_rot15/（aug_rot15）。
- 狀態：已於 2026-10-27 執行完成（test mAP50 0.711；person 0.852、dynamic_obstacle 0.57；與 run02 差異在單一 seed 波動範圍內，結論見檔尾）。

In [ ]:
# [步驟1] 確認 GPU 型號（T4），並安裝 ultralytics（YOLOv8）與 roboflow 套件
!nvidia-smi -L && pip -q install ultralytics roboflow

GPU 0: Tesla T4 (UUID: GPU-8994e472-27f3-0423-2d38-2322feef4b63)


In [ ]:
# [步驟2] 下載 Roboflow alert-yolo-v1 的 version 2（v2-raw-640-5class-fix0524，5 類，無擴增）
from google.colab import userdata; from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace('ssudscompanionrobot').project('alert-yolo-v1').version(2).download('yolov8')
print(ds.location)

loading Roboflow workspace...
loading Roboflow project...
/content/alert-yolo-v1-2


In [ ]:
# [步驟3] 標籤合併：car/bus/motorcycle/bicycle -> dynamic_obstacle（id=1），person -> id=0；只改標籤檔，影像不動，輸出到 /content/ds_dyn
import yaml, shutil, glob, os
src = ds.location; dst = '/content/ds_dyn'
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
names = yaml.safe_load(open(src + '/data.yaml'))['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
new_id = {i: (0 if n == 'person' else 1) for i, n in enumerate(names)}
cnt = {}
for f in glob.glob(dst + '/*/labels/*.txt'):
    split = f.split('/')[-3]; out = []
    for line in open(f).read().splitlines():
        p = line.split()
        if len(p) < 5: continue
        p[0] = str(new_id[int(p[0])]); out.append(' '.join(p))
        cnt[(split, p[0])] = cnt.get((split, p[0]), 0) + 1
    open(f, 'w').write('\n'.join(out) + ('\n' if out else ''))
cfg = yaml.safe_load(open(dst + '/data.yaml'))
cfg.update(path=dst, train=dst + '/train/images', val=dst + '/valid/images', test=dst + '/test/images', nc=2, names=['person', 'dynamic_obstacle'])
yaml.safe_dump(cfg, open(dst + '/data.yaml', 'w'))
print(names, '->', ['person', 'dynamic_obstacle'])
print('各切分實例數 (split, 類別id): ', dict(sorted(cnt.items())))

['bicycle', 'bus', 'car', 'motorcycle', 'person'] -> ['person', 'dynamic_obstacle']
各切分實例數 (split, 類別id):  {('test', '0'): 103, ('test', '1'): 31, ('train', '0'): 508, ('train', '1'): 158, ('valid', '0'): 81, ('valid', '1'): 15}


In [ ]:
# [步驟4] 與 run02 相同設定訓練 2 類模型，唯一差異：degrees=15（隨機旋轉 ±15°，只作用於訓練集；mosaic/hsv/fliplr/scale/translate 仍為 0），再用 best.pt 在 test 集評估
from ultralytics import YOLO
m = YOLO('yolov8n.pt')
m.train(data=dst + '/data.yaml', imgsz=320, epochs=100, patience=20, batch=32, seed=0, deterministic=True, mosaic=0, hsv_h=0, hsv_s=0, hsv_v=0, fliplr=0, scale=0, translate=0, degrees=15, weight_decay=0.0005, project='runs', name='aug_rot15', plots=False, verbose=False)
best = YOLO(str(m.trainer.best))
r = best.val(data=dst + '/data.yaml', split='test', imgsz=320, plots=False)
print('TEST mAP50', r.box.map50, 'mAP50-95', r.box.map, 'P', r.box.mp, 'R', r.box.mr)
print({r.names[int(i)]: (round(float(a), 3), int(r.nt_per_class[int(i)])) for i, a in zip(r.box.ap_class_index, r.box.ap50)})

Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ds_dyn/data.yaml, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=aug_rot15, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, ove

In [ ]:
# [步驟5] 掛載 Google Drive，把本次訓練結果複製到 capstone_yolo/run03_v2dynamic2class_rot15/
from google.colab import drive; drive.mount('/content/drive')
import shutil, os
out = '/content/drive/MyDrive/capstone_yolo/run03_v2dynamic2class_rot15'
os.makedirs(out, exist_ok=True)
shutil.copytree('/content/runs/detect/runs/aug_rot15', out + '/aug_rot15', dirs_exist_ok=True)
print(os.listdir(out))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['aug_rot15']


# 本版結論（run03：旋轉 ±15°）
- **結果**：test mAP50 = 0.711、mAP50-95 = 0.321、P = 0.817、R = 0.643；person AP50 = 0.852（P 0.916、R 0.709）、dynamic_obstacle AP50 = 0.57（P 0.719、R 0.578）。第 40 epoch 提前停止，最佳為第 20 epoch。
- **與 run02 比較**（seed=0；括號為 run02）：mAP50 0.711（0.688）+0.023；mAP50-95 0.321（0.367）-0.046；person 0.852（0.866）-0.014；dynamic_obstacle 0.57（0.51）+0.06；Precision 0.817（0.719）上升、Recall 0.643（0.679）下降。
- **判讀**：mAP50 與 person 的差距都在約 0.03 以內，落在單一 seed 可能的波動範圍（run02 尚無多 seed 變異估計）；dynamic_obstacle 的 +0.06 只來自 31 個 test 實例（召回由 0.495 升到 0.578，約多找到 2～3 個目標），不足以下結論。mAP50-95 下降 0.046 是這次最明顯的變化，代表框的定位精度可能變差，但同樣僅單次結果。
- **假設（未驗證）**：旋轉擴增會把框放大成旋轉後的外接框，可能使框變鬆、拉低高 IoU 門檻下的指標；需以多 seed 與檢視預測框確認。
- **結論**：目前沒有證據顯示旋轉 ±15° 有效，也沒有證據顯示有害；不採用也不排除，暫列為「待多 seed 驗證」。run02 仍是正式 raw baseline。
- **限制**：單一 seed；test 的 dynamic_obstacle 僅 31 個；兩次實驗皆提前停止（最佳 epoch 30 與 20），後段 epoch 的波動也可能影響結果。
- **下一步**：(1) 以 3 個 seed 重跑 run02 與 run03 設定，取平均與標準差再比較；(2) 若有跡象再做 ±30°、±45° 與 HSV；(3) 補資料後以新資料版本重做 baseline 與擴增比較，作為正式結論。
- **輸出位置**：Google Drive/capstone_yolo/run03_v2dynamic2class_rot15/aug_rot15（已確認複製完成）。